<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="./images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="20%">
</div>

<br>

# INTERACTIVE VISUALIZATIONS WITH PLOTLY AND DASH

<br>

**About:** Build interactive charts with Plotly and turn them into shareable web dashboards using Dash.

**Learning Goals:**
1. Distinguish static, interactive, and dynamic visualization and identify when interactivity adds value.
2. Construct interactive Plotly charts (box plots, scatter overlays, correlation heatmaps) from tabular data.
3. Understand the layout-and-callback model that makes a Dash application reactive.
4. Run a local Dash app and connect a dropdown widget to a live chart update.

**Keywords:** plotly, dash, interactive visualization, dashboards, callbacks, python

**Prerequisite Knowledge:** (1) Basic Python and pandas - see `01_visualization_principles_matplotlib_seaborn.ipynb`; (2) Familiarity with static charting (matplotlib or seaborn)

**Target User:** Python learners who can read and run pandas code and want to move beyond static plots into browser-based interactive visualization.

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: STATIC VS INTERACTIVE - WHEN DOES INTERACTIVITY HELP?](#Part_0)
> #### [PART 1: INTERACTIVE CHARTS WITH PLOTLY](#Part_1)
> #### [PART 2: BUILDING A DASH APPLICATION](#Part_2)

#### APPENDIX

> #### [APPENDIX I: Additional Interactive Libraries](#Appendix_I)

<br>

<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **STATIC** vs **INTERACTIVE** - When Does Interactivity Help?

Static charts - PNG or PDF output - are the default output of matplotlib and seaborn. They are sufficient when the message is fixed and the audience is a reader who cannot interact with the output (a printed report, a slide deck, an academic paper).

Interactive charts add value when:

- **The data has more dimensions than a static view can show.** A heatmap that shows a correlation matrix becomes more informative when hovering over a cell reveals the exact coefficient and the two variable names.
- **The audience needs to filter or explore.** A time-series that covers ten years cannot show detail for every year at once; a slider or dropdown lets the reader focus.
- **The output lives in a browser or notebook.** If the consumer will click it, build for clicking.

When interactivity does not add value: when the conclusion is simple enough to state in a title; when the audience is non-technical and the interaction itself would confuse rather than clarify; when the deployment environment cannot render JavaScript (PDF export, print).

___

**Note:** "Interactive" and "dynamic" are often used interchangeably but mean different things here. An interactive chart responds to the user's mouse (hover, click, zoom). A dynamic chart updates its data from an external source on a schedule without the user doing anything (a live dashboard pulling from a database every 30 seconds). Plotly handles interactivity; Dash handles both.

___

**Sources consulted:** Plotly documentation - https://plotly.com/python/; "Visualization Analysis and Design" by Tamara Munzner (2014), Chapter 11 (Manipulate).

<!--Concept Check-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **You have a dataset of monthly sales figures for 12 product categories across 5 years. A manager wants to explore seasonal trends for any category she picks. Would you build a static chart or an interactive one? What specific interaction - hover, filter, zoom, or something else - would add the most value for her task?**

<br>

```python
# Write your answer as a comment below.
# Answer:
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **INTERACTIVE CHARTS** with **PLOTLY**

<br>

Plotly is a Python charting library that produces browser-native interactive figures. Every Plotly chart supports hover tooltips, zoom, pan, and legend toggling out of the box - no extra code required beyond the chart definition itself. Plotly figures can be displayed inline in Jupyter, exported to standalone HTML files, or embedded in a Dash application (Part 2).

This Part builds two chart types from real survey data (the BII dataset in `data/`):
- A box-and-scatter overlay showing the distribution of each feature score
- A correlation heatmap showing pairwise relationships between features

#### CONTENTS:

> [PART 1.1: BOX + SCATTER AND CORRELATION HEATMAP](#Part_1_1)<br>

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: BOX + SCATTER AND CORRELATION HEATMAP

<br>

The BII (Behavioral Intelligence Index) dataset in `data/bii_data_w_categories.csv` contains survey scores for traits like trust, resilience, and collaboration across individuals. The two charts below reveal both the spread of each trait (box + scatter) and how traits correlate with each other (annotated heatmap).

**Why Plotly for these charts?** Matplotlib can produce both chart types, but the hover tooltips Plotly adds for free make the correlation heatmap genuinely more readable: instead of squinting at annotated coefficient numbers, the reader hovers over any cell and reads the exact values.

___

**Note:** Volatile API - Plotly's `graph_objects` and `figure_factory` interfaces were stable through Plotly 5.x (verified 2026-08-25). Re-check import names at https://plotly.com/python/figure-factories/ if you encounter ImportError.

___

In [ ]:
%ls


In [ ]:
# (Working directory is the repo root - no need to change it)

In [ ]:
# import os; print(os.getcwd())  # uncomment to verify working directory

In [ ]:
# Remove warnings and display plots inline
import matplotlib as aplt
%matplotlib inline

import matplotlib as mpl
mpl.rcParams['axes.linewidth'] = 0.5 #set the value globally

import warnings
warnings.filterwarnings('ignore') # remove warnings

In [ ]:
# Import background libraries
from pylab import *
import pandas as pd
import numpy as np
import time

# Import Plotly related libraries
import plotly.tools as tls
import plotly.offline as offline
from plotly.graph_objs import *
import plotly.graph_objects as go
import plotly.figure_factory as ff
from scipy.spatial import Delaunay
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot

# pretty colors
import seaborn as sns
import colorlover as cl
import pprint as pp

# Optional: Chart Studio credentials for online publishing.
# Sign up free at https://chart-studio.plotly.com then uncomment:
# import chart_studio
# chart_studio.tools.set_credentials_file(username='YOUR_USERNAME', api_key='YOUR_API_KEY')
from IPython.display import HTML


In [ ]:
# generating plots within Jupyter
init_notebook_mode(connected=True)

In [ ]:
# load data
df = pd.read_csv("./data/bii_data_w_categories.csv")
df.head()

In [ ]:
#### DESIRED ARTISTS ####

client = "My Organization"  # replace with your organization name
workgrp = " Visualizations"

# create sequential color scales
c_scale = [
    [0.0, 'rgba(38, 82, 112, 0.8)'], 
    [0.5, 'rgba(78, 94, 110, 1.0)'],
    [1.0, 'rgba(247, 127, 3, 0.8)']
]

heat_scale = [
    [0.0, 'rgba(255, 255, 255, 0.1)'],
    [0.2, 'rgba(38, 82, 112, 0.3)'],
    [0.4, 'rgba(38, 82, 112, 0.5)'],
    [0.6, 'rgba(78, 94, 110, 0.7)'],
    [0.9, 'rgba(78, 94, 110, 0.9)'],
    [1.0, 'rgba(247, 127, 3, 1.0)']
]

heat_scale_2 = [
    [0.0, 'rgba(255, 255, 255, 0.7)'],
    [0.5, 'rgba(78, 94, 110, 1.0)'],
    [1.0, 'rgba(247, 127, 3, 0.9)']
]

# Set individual feature colors
col = 'rgba(143, 152, 161, 0.9)'
tru = 'rgba(106, 120, 133, 0.9)'
per = 'rgba(78,  94,  110, 0.9)'
iz  = 'rgba(56,  72,  87,  0.9)'
cz  = 'rgba(4,   36,  60,  0.9)'
div = 'rgba(105, 138, 163, 0.9)' 
bel = 'rgba(65,  105, 136, 0.9)'
res = 'rgba(39,  80,  112, 0.9)'
bii = 'rgba(247, 127, 3,   1.0)'
box_scale = [tru,res, div, bel, per, col, cz, iz, bii]

# other potential fill colors
fill = 'rgb(78, 94, 110)' # orange box fill
fill2 = 'rgb(38, 82, 112)'
fill3 = 'rgb(78, 94, 110)'
fill4 = 'rgb(78, 94, 110)'
fill5 = 'rgb(247, 127, 3)'

# set font colors
font_colors = ['#275170','#4E5E6E','#4E5E6E', '#FFFFFF']

In [ ]:
#### SET LOCATION AND NAMES FOR OFFLINE PLOTS ####

# Relative path - plot HTML files are saved alongside the notebook
path = "./data/plot_images/"

import os
os.makedirs(path, exist_ok=True)

# name files
corr_plot = "Feature Correlation (Heat)"
boxS_plot = "Box and Scatter"
corr_file_name = path + corr_plot + "_" + client + "_" + workgrp
boxS_file_name = path + boxS_plot + "_" + client + "_" + workgrp


In [ ]:
#sanity check
df.head(3)

In [ ]:
#### CREATE SUBSETS USED FOR PLOTTING ####

# create list of features for processing
feat_cols = ['trust', 'resilience', 'diversity', 'belief', 'perfection',
       'collaboration', 'comfort zone', 'innovation zone', 'bii score']

# select feature and data (leave response variable data)
y_data = df[feat_cols[:-1]]
x_data = feat_cols

# remove response variable form feature list
res_cols2 = feat_cols[:-1]

# compute mean and standard div
feat_mean = df[res_cols2].mean()
feat_std = df[res_cols2].std()

In [ ]:
#### ITERATIVELY CREATE TRACES USED IN PLOTTING ####

# Intiate list to be used for iterative axes creation
traces = []


for xd, yd, cols in zip(x_data, y_data, box_scale):
        traces.append(go.Box(
            y = y_data[yd],
            name = xd,
            hoverlabel = dict(
                bgcolor = '#EFEFEF', 
                bordercolor = cols),
            boxpoints = 'all',
            boxmean = True,
            width = .5,
            jitter = .5,
            whiskerwidth=0.2,
            fillcolor=fill2,        # use fill for single color on all boxes
            marker_size=6,
            marker = dict(
                symbol  = 'circle',
                opacity = 0.2,
                size    = 14,
                color   = '#E26130',
                line = dict(
                    color = '#E26130',
                    width = 2
                )
            ),
            line = dict(
                color= '#E26130',
                width = 1
            ),
        )
                     )


In [ ]:
#### UPDATE ARTISTS USING LAYOUT: MAKE PRETTY ####

layout = go.Layout(
    font = dict(
        family = 'Open Sans',
        size = 11
    ),
    #title =  'Mean of Feature Scores for %s' % client,
    title = 'Feature Scores (Mean) ' + client,
    width=900,
    height=600,
    paper_bgcolor = 'rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    boxmode='group',
    yaxis=dict(
        title = 'Scores',
        autorange=True,
        showgrid=True,
        zeroline=True,
        dtick=5,
        gridcolor='#EFEFEF',
        gridwidth=1,
        zerolinecolor='#EFEFEF',
        zerolinewidth=2
    ),
    xaxis = dict(
        title = 'Features'
    ),
    
    margin=dict(
        l=40,
        r=30,
        b=80,
        t=100,
    ),



    showlegend=False
)
    

In [ ]:
#### BRING IT ALL TOGETHER: PLOT FIGURE ####
fig = go.Figure(data=traces, layout=layout)
offline.plot(fig, image='png',filename = boxS_file_name, show_link=False)

In [ ]:
#### VIEW RESULTS ####
HTML(boxS_file_name + ".html")


<br>

___

<br>

In [ ]:
#### CORRELATION HEATMAP ####
fig = ff.create_annotated_heatmap(z=df[res_cols2].corr().round(2).values, 
                                  x=res_cols2, y=res_cols2, 
                                  colorscale=heat_scale_2, font_colors = font_colors, name='Feature Correlation Heatmap')

#fig.layout.title = 'Individual Trait Correlations for %s' % client
fig.layout.title = 'Individual Train Correlations ' + client
#fig.layout.font = {'family': 'Open Sans', 'size':14, 'color':'#fff'}  #font color set to white
fig.layout.font = {'family': 'Open Sans', 'size':14}                   #defalt font color

fig.layout.xaxis = {'showgrid': False}
fig.layout.yaxis = {'showgrid': False, 'tickangle':45}
fig.layout.margin = {'l':90}
fig.layout.paper_bgcolor = 'rgba(0,0,0,0)'
fig.layout.plot_bgcolor='rgba(0,0,0,0)'


offline.plot(fig, image='png',filename=corr_file_name, show_link=True, image_width=800, image_height=800)

In [ ]:
#### VIEW RESULTS ####
HTML(corr_file_name + ".html")


<br>

___

<br>

In [ ]:
# resilience, diversity, belief
cor_df = feat_cols[:-4]

resilience = 'Disciplined, Gritty, Focused, Experimental, Agile'
diversity = 'Partner, Experiential, Open, Networker, Outgoing'
belief = 'Focused, Goal-Driven, Quantitative, Deeply Curious, Adaptable'

symbol = [
    ['', ''                      , ''                      , ''                   , ''],
    ['', 'Belief + Resilience'   , 'Belief + Diversity'    , 'Belief'             , ''],
    ['', 'Diversity + Resilience', 'Diversity'             , 'Diversity + Belief' , ''],
    ['', 'Resilience'            , 'Resilience + Diversity', 'Resilience + Belief', ''],
    ['', ''                      , ''                      , ''                   , '']
]
br = belief + resilience
bd = belief + diversity
dr = diversity + resilience

element = [
    ['', '',         '',         '',         ''],
    ['', belief,     belief,     belief    , ''],
    ['', diversity,  diversity,  diversity , ''],
    ['', resilience, resilience, resilience, ''],
    ['', '',         '',         '',         '']
]
atomic_mass = [
    ['', '',         '',        '',     ''],
    ['', resilience, diversity, '',     ''],
    ['', diversity,  '',        belief, ''],
    ['', '',         diversity, belief, ''],
    ['', '',         '',        '',     '']
]

heat_scale = [
    [0.0, 'rgba(255, 255, 255, 0.1)'],
    [0.1, 'rgba(38,  82,  112, 0.1)'],
    [0.4, 'rgba(38,  82,  112, 0.2)'],
    [0.5, 'rgba(78,  94,  110, 0.5)'],
    [0.7, 'rgba(78,  94,  110, 0.9)'],
    [1.0, 'rgba(247, 127, 3,   1.0)']
]

# Display element descriptive words on hover
hover=range(len(symbol))
for x in range(len(symbol)):
    hover = list(hover)
    hover[x] = ['<br>' + 'Descriptive Words: ' + '<br>'+ i + '<br>' + str(j) for i, j in zip(atomic_mass[x], element[x])]

# Invert Matrices
symbol = symbol[::-1]
hover = hover[::-1]
z = cor_df[::-1]    


In [ ]:
# file name for new plot
file_name = corr_file_name + "_annotated"

In [ ]:
#### PLOT ANNOTATED CORRELATION HEATMAP ####
# plot correlation heatmap
fig = ff.create_annotated_heatmap(z=df[z].corr().round(2).values, x=cor_df, y=cor_df,
                                  colorscale=heat_scale, font_colors = font_colors, 
                                  name=file_name, annotation_text=symbol, text=hover)

offline.init_notebook_mode()

fig.layout.title = 'Feature Heatmap - %s' % client
fig.layout.font = {'family': 'Open Sans', 'size':12}
fig.layout.xaxis = {'visible': False}
fig.layout.yaxis = {'visible': False}
offline.plot(fig, image='png',filename=file_name, show_link=False, image_width=500, image_height=400)

In [ ]:
HTML(file_name + ".html")


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#0D0D0D; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<!--Concept Check-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The correlation heatmap uses `ff.create_annotated_heatmap`. Look at the `colorscale` argument passed to it. What does the color mapping from white to blue to orange communicate to a reader? Rewrite the colorscale so that negative correlations appear in red, near-zero correlations appear in white, and positive correlations appear in blue. You do not need to run it - just write the list.**

<br>

```python
# Write your revised colorscale here:
revised_heat_scale = [
    [0.0, '...'],   # negative extreme
    [0.5, '...'],   # near-zero
    [1.0, '...'],   # positive extreme
]
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **BUILDING** a **DASH** Application

<br>

Plotly charts are interactive as standalone figures. Dash takes the next step: it lets you wrap one or more Plotly charts in a web application that responds to user input through Python callbacks - no JavaScript required.

A Dash app has three components:

1. **The app object** - `app = dash.Dash(__name__)` creates the application.
2. **The layout** - An HTML tree (built with `html.*` and `dcc.*` components) that defines what appears on the page. Think of it as the static scaffold.
3. **Callbacks** - Python functions decorated with `@app.callback` that fire when a user changes a control (a dropdown, slider, or button) and update one or more output components. This is the reactive part.

#### CONTENTS:

> [PART 2.1: APP STRUCTURE AND LAYOUT](#Part_2_1)<br>
> [PART 2.2: ADDING CALLBACKS](#Part_2_2)<br>

<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: APP STRUCTURE AND LAYOUT

<br>

The `DASH_Plotly/dash_intro.py` file in this repo is a minimal but complete Dash application. Reading it alongside this explanation gives you the full picture.

___

**Note:** Volatile API - Dash's import structure changed significantly in Dash 2.0 (2021). `dash_core_components` and `dash_html_components` were merged into `dash` itself. The imports below use the Dash 2.x style (verified 2026-08-25 against https://dash.plotly.com/installation). If you see `ModuleNotFoundError: No module named 'dash_core_components'`, upgrade Dash.

___

In [ ]:
# Install Dash if needed
# pip install dash plotly pandas

# Verify versions - Dash 2.x required
import dash
import plotly
print(f"Dash version: {dash.__version__}")
print(f"Plotly version: {plotly.__version__}")

In [ ]:
# A minimal Dash app - run this as a .py file, not in Jupyter
# (Jupyter-in-Dash requires JupyterDash; this illustrates the structure)

import pandas as pd
import plotly.graph_objects as go
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

# 1. Create the app object
app = dash.Dash(__name__)

# 2. Load data
df = pd.read_csv("./data/bii_data_w_categories.csv")
feat_cols = ['trust', 'resilience', 'diversity', 'belief', 'perfection',
             'collaboration', 'comfort zone', 'innovation zone']

# 3. Define the layout - HTML tree with a dropdown and a chart placeholder
app.layout = html.Div([
    html.H1("BII Feature Distribution Explorer", style={'text-align': 'center'}),
    
    dcc.Dropdown(
        id="feature_selector",
        options=[{"label": col.title(), "value": col} for col in feat_cols],
        value="trust",        # default selection
        clearable=False,
        style={'width': '40%'}
    ),
    
    dcc.Graph(id="feature_chart")  # placeholder - filled by callback
])

# The callback (Part 2.2) connects the dropdown to the chart
if __name__ == '__main__':
    app.run(debug=True)

**What each piece does:**

- `html.Div([...])` - the root container; everything on the page lives inside it.
- `html.H1(...)` - a static heading, rendered as `<h1>` in the browser.
- `dcc.Dropdown(id="feature_selector", ...)` - an interactive dropdown. The `id` is how the callback refers to it.
- `dcc.Graph(id="feature_chart")` - an empty chart container. The `id` connects it to the callback's `Output`.

The layout describes *what* is on the page. The callback (Part 2.2) describes *how it changes* when the user interacts.

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.2: ADDING CALLBACKS

<br>

A callback is a Python function that Dash calls automatically whenever an `Input` component changes. The decorator `@app.callback(...)` registers the function and declares what it reads (`Input`) and what it writes (`Output`).

The signature matches: one argument per `Input`, one return value per `Output`.

In [ ]:
# Add this callback to the app defined in Part 2.1
# Place it between the layout definition and app.run()

@app.callback(
    Output(component_id="feature_chart", component_property="figure"),
    Input(component_id="feature_selector", component_property="value")
)
def update_chart(selected_feature):
    """
    Fires when the dropdown changes.
    selected_feature: the 'value' of the chosen dropdown option (a column name).
    Returns: a Plotly figure dict, assigned to 'figure' property of dcc.Graph.
    """
    fig = go.Figure()
    
    fig.add_trace(go.Box(
        y=df[selected_feature],
        name=selected_feature.title(),
        boxpoints='all',
        jitter=0.3,
        pointpos=-1.8,
        marker_color='#003262'
    ))
    
    fig.update_layout(
        title=f"Distribution of {selected_feature.title()} Scores",
        yaxis_title="Score",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)'
    )
    
    return fig

**Why the callback model matters:**

Every time the user picks a different feature in the dropdown, Dash calls `update_chart` with the new value. The function returns a new figure, and the `dcc.Graph` updates in the browser - without reloading the page and without any JavaScript on your part.

This is the reactive pattern: `Input` triggers, `Output` receives. More complex apps chain callbacks together or have multiple inputs and outputs per callback, but the pattern is always the same.

**Running the app:**

Save the full script (layout + callback) as `my_dash_app.py` and run:

```bash
python my_dash_app.py
```

Open `http://127.0.0.1:8050` in your browser. The `debug=True` flag enables hot-reload: save the file and the browser refreshes automatically.

___

**Note:** Jupyter and Dash do not share the same execution model. Running `app.run()` inside a Jupyter cell blocks the kernel. Use `jupyter_dash` (`pip install jupyter-dash`) if you want to prototype inside a notebook. The standalone script is the recommended pattern for sharing a finished app.

___

<!--Concept Check-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Without running the code: if you add a second `dcc.Dropdown` to the layout - call it `"group_selector"` - that lets the user filter the dataset by category group, what would change in the `@app.callback` decorator and in the function signature? Write the updated decorator and function signature (not the body).**

<br>

```python
# Write the updated decorator and function signature:
@app.callback(
    Output(...),
    ...
)
def update_chart(...):
    pass
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Appendix_I'></a>

<hr style="border: 2px solid#003262;" />

## APPENDIX I: Additional Interactive Libraries

> * **Bokeh** - A library with a similar philosophy to Plotly but different API conventions. Strong for streaming data. https://bokeh.org
> * **Panel (HoloViews)** - Dashboard framework built on Bokeh. Closer to Jupyter-native than Dash. https://panel.holoviz.org
> * **Streamlit** - Fast prototyping of data apps with a script-runs-top-to-bottom model. Less flexible than Dash but much faster to get running. https://streamlit.io
> * **Vega-Lite / Altair** - A grammar-of-graphics approach to interactive charts. Declarative syntax closer to SQL than Python imperative code. https://altair-viz.github.io

<hr style="border: 6px solid#003262;" />